# Tin tức (Unstructured Data) — Pipeline v2

Notebook điều khiển luồng tin tức **3 tầng**: Discovery → Detail Fetch → Canonical Articles.

| Tầng | Mô tả |
|------|-------|
| **Discovery** | Thu thập link + title + snippet từ vnstock / RSS / HTML list |
| **Detail Fetch** | Crawl trang bài viết (VnExpress, CafeF) lấy `body_text` thật |
| **Canonical Articles** | Merge thành dataset sạch cho search / embedding / analysis |

| Đầu ra | Đường dẫn |
|--------|----------|
| Discovery | `news/discovery/date=<ngày>/PART-000.parquet` |
| Articles | `news/articles/date=<ngày>/PART-000.parquet` |
| (compat) vnstock/rss/html | `news/<source>/date=<ngày>/PART-000.parquet` |

In [1]:
import os, sys, warnings, threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_orig_hook = threading.excepthook
def _quiet_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass
    else:
        _orig_hook(args)
threading.excepthook = _quiet_hook
warnings.filterwarnings("ignore")

from pathlib import Path
root = Path.cwd()
if not (root / "ingestion").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from ingestion.common import configure_logging, load_dotenv_from_project_root
from ingestion.unstructured_data import (
    NewsIngestionConfig,
    ingest_news,
    primary_news_display_path,
    validate_articles,
    validate_discovery,
    report_articles,
)

configure_logging()
load_dotenv_from_project_root()
print("[OK] setup")

2026-04-16 13:02:17 [INFO] Đã nạp biến môi trường từ C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\.env


[OK] setup


## Cấu hình

In [2]:
news_cfg = NewsIngestionConfig(
    use_listing_tickers=True,
    listing_exchange_filter=["HSX", "HNX"],
    max_tickers_per_run=50,
    max_articles_per_source=200,
    days_back=0,
    enable_vnstock=True,
    enable_rss=True,
    enable_html=True,
    # ── Detail-fetch (Tier 2) ──
    enable_detail_fetch=True,
    max_detail_fetch_per_run=200,
    detail_min_body_length=200,
    output_discovery=True,
    output_articles=True,
)

print(f"run_date           : {news_cfg.run_date}")
print(f"news_root          : {news_cfg.news_root}")
print(f"sources.yaml       : {news_cfg.resolved_sources_yaml()}")
print(f"listing            : {news_cfg.resolved_listing_parquet()}")
print(f"enable_detail_fetch: {news_cfg.enable_detail_fetch}")
print(f"max_detail_fetch   : {news_cfg.max_detail_fetch_per_run}")

run_date           : 2026-04-16
news_root          : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news
sources.yaml       : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\ingestion\unstructured_data\sources.yaml
listing            : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet
enable_detail_fetch: True
max_detail_fetch   : 200


## Chạy

In [3]:
news_paths = ingest_news(news_cfg)
news_paths

2026-04-16 13:02:17 [INFO] Đã nạp biến môi trường từ C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\.env
2026-04-16 13:02:22 [INFO] API key setup completed
2026-04-16 13:02:22 [INFO] Đã gọi register_user từ VNSTOCK_API_KEY.


✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! vnst***6437
✓ Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)


2026-04-16 13:02:24 [INFO] News listing: sau lọc sàn ['HSX', 'HNX'] còn 705 mã (lấy 50).
2026-04-16 13:02:24 [INFO] News: dùng 50 mã từ listing (tối đa max_tickers_per_run=50).
2026-04-16 13:02:34 [INFO] VnstockNewsAdapter [KBS]: AAA @kbs -> 4 discoveries
2026-04-16 13:02:40 [INFO] VnstockNewsAdapter [KBS]: AAM @kbs -> 1 discoveries
2026-04-16 13:02:48 [INFO] VnstockNewsAdapter [KBS]: AAT @kbs -> 3 discoveries
2026-04-16 13:03:00 [INFO] VnstockNewsAdapter [KBS]: AAV @kbs -> 6 discoveries
2026-04-16 13:03:06 [INFO] VnstockNewsAdapter [KBS]: ABR @kbs -> 1 discoveries
2026-04-16 13:03:14 [INFO] VnstockNewsAdapter [KBS]: ABS @kbs -> 3 discoveries
2026-04-16 13:03:20 [INFO] VnstockNewsAdapter [KBS]: ABT @kbs -> 1 discoveries
2026-04-16 13:03:46 [INFO] VnstockNewsAdapter [KBS]: ACB @kbs -> 45 discoveries
2026-04-16 13:03:54 [INFO] VnstockNewsAdapter [KBS]: ACC @kbs -> 3 discoveries
2026-04-16 13:04:04 [INFO] VnstockNewsAdapter [KBS]: ACG @kbs -> 6 discoveries
2026-04-16 13:04:10 [INFO] Vnsto

{'vnstock': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\vnstock\\date=2026-04-16\\PART-000.parquet',
 'rss': '',
 'html': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\html\\date=2026-04-16\\PART-000.parquet',
 'combined': '',
 'discovery': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\discovery\\date=2026-04-16\\PART-000.parquet',
 'articles': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\articles\\date=2026-04-16\\PART-000.parquet'}

## Kiểm tra kết quả

In [4]:
import pandas as pd

print("=" * 70)
print("OUTPUT FILES")
print("=" * 70)
for key, path in news_paths.items():
    if not path:
        print(f"  {key:12s}: (trống)")
        continue
    df = pd.read_parquet(path)
    print(f"  {key:12s}: {len(df):>5d} dòng  →  {path}")

print(f"\nPrimary display path: {primary_news_display_path(news_paths)}")

OUTPUT FILES
  vnstock     :     1 dòng  →  C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\vnstock\date=2026-04-16\PART-000.parquet
  rss         : (trống)
  html        :    60 dòng  →  C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\html\date=2026-04-16\PART-000.parquet
  combined    : (trống)
  discovery   :   108 dòng  →  C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\discovery\date=2026-04-16\PART-000.parquet
  articles    :    61 dòng  →  C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\articles\date=2026-04-16\PART-000.parquet

Primary display path: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\articles\date=2026-04-16\PART-000.parquet


## Validation & Data Quality

In [5]:
# Validate discovery output
if news_paths.get("discovery"):
    disc_df = pd.read_parquet(news_paths["discovery"])
    print(f"Discoveries: {len(disc_df)} rows")
    issues = validate_discovery(disc_df)
    if issues:
        print("⚠ Discovery issues:")
        for i in issues:
            print(f"  - {i}")
    else:
        print("✓ Discovery validation passed")
    print(f"\nSource types: {disc_df['source_type'].value_counts().to_dict()}")
    print(f"Sources:      {disc_df['source'].value_counts().to_dict()}")
else:
    print("No discovery output.")

Discoveries: 108 rows
⚠ Discovery issues:
  - 1 rows with empty canonical_url

Source types: {'rss': 60, 'html_list': 47, 'vnstock': 1}
Sources:      {'vnexpress.net': 60, 'vnexpress': 47, 'vnstock_kbs': 1}


In [6]:
# Validate articles output
if news_paths.get("articles"):
    art_df = pd.read_parquet(news_paths["articles"])
    print(f"Articles: {len(art_df)} rows\n")

    issues = validate_articles(art_df)
    if issues:
        print("⚠ Article issues:")
        for i in issues:
            print(f"  - {i}")
    else:
        print("✓ Article validation passed")

    print("\n" + report_articles(art_df))

    print("\n--- Sample 3 rows with body_text ---")
    has_body = art_df[art_df["body_text"].astype(str).str.len() > 100]
    for _, row in has_body.head(3).iterrows():
        print(f"\n[{row['content_type']}] {row['title'][:80]}")
        print(f"  source: {row['source']}  |  url: {row['canonical_url'][:60]}...")
        print(f"  body_text ({len(str(row['body_text']))} chars): {str(row['body_text'])[:200]}...")
else:
    print("No articles output.")

Articles: 61 rows

✓ Article validation passed

Total articles: 61
  content_type=article: 60
  content_type=snippet: 1
body_text stats: min=0, median=2649, max=5872
Rows with non-empty body_text: 60 (98.4%)

Top 5 shortest articles (with body):
  [ 1162 chars] BIDV kết nối hệ thống chuyển mạch ngành chứng khoán...
  [ 1535 chars] Ông Nguyễn Văn Thời xin từ nhiệm Chủ tịch Dệt may TNG...
  [ 1609 chars] Lợi nhuận của TCBS tiếp tục tăng trưởng nhờ mảng cho vay...
  [ 1637 chars] VPBankS trở thành cổ đông lớn tại công ty của ông Đặng Thành...
  [ 1652 chars] Nhân viên tại HSBC thu nhập bình quân gần 1 tỷ đồng một năm...

Top 5 longest articles:
  [ 5872 chars] ABBank lên kế hoạch tăng tốc trong năm 2026...
  [ 5798 chars] Bài toán thu - chi của tân Bộ trưởng Tài chính...
  [ 5464 chars] Tân Thống đốc đối mặt bài toán 'kép' tăng trưởng và an toàn ...
  [ 5267 chars] Sếp Novaland khẳng định công ty có tài sản nhiều hơn nợ...
  [ 5242 chars] Chật vật đánh thức mỏ đất hiếm lớn nhất châu Âu...